In [3]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import re

In [4]:
params_data = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=Ventas_Shopify;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor de conexión
engine_data = create_engine(f"mssql+pyodbc:///?odbc_connect={params_data}")

# Configurar cadena de conexión
params_com = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=comerssiamirror.eastus2.cloudapp.azure.com,38693;"
    "DATABASE=PROVENZAL;"
    "UID=provenzal;"
    "PWD=PE@4:BRy/<1VWkp;"
)

# Crear el motor
engine_com = create_engine(f"mssql+pyodbc:///?odbc_connect={params_com}")

query_data = """
SELECT DISTINCT 
    c.Id AS Cliente,
    v.Fecha AS Fecha_Venta,
    v.Canal,
	h.Categoria,
	h.Venta_Neta AS Venta
FROM Contacto_Clientes c
INNER JOIN Ventas_ShopifyMoviNova v ON c.id = v.id_cliente
INNER JOIN Venta_Homologada h ON v.Pedido = h.Pedido
WHERE v.fecha >= '2025-06-01'
"""

# Ejecutar y cargar en DataFrame
ventas_shopify = pd.read_sql(query_data, engine_data)

query_com = """
SELECT DISTINCT 
    c.CLICodigo AS Cliente,
    v.Fecha AS Fecha_Venta,
    v.Canal,
    v.Categoria,
    v.Venta_Neta AS Venta
FROM Contacto_Clientes c
INNER JOIN Ventas_MoviNova v ON c.CLICodigo = v.Cliente
WHERE v.fecha >= '2025-06-01'
"""

# Ejecutar y cargar en DataFrame
ventas_comerssia = pd.read_sql(query_com, engine_com)

In [5]:
mapa_categorias = {
    '00002': 'ARTICULOS DE ASEO',
    '00003': 'CUIDADO CAPILAR',
    '00004': 'CUIDADO CORPORAL',
    '00005': 'CUIDADO FACIAL',
    '00006': 'CUIDADO DE MANOS',
    '00007': 'HOGAR',
    '00009': 'OTROS',
    '00010': 'FRAGANCIAS',
    '00016': 'KIT MULTICATEGORÍA'
}

ventas_comerssia['Categoria'] = ventas_comerssia['Categoria'].map(mapa_categorias)


df_ventas = pd.concat([ventas_comerssia, ventas_shopify], ignore_index=True)


In [6]:
def limpiar_codigo(codigo):
    codigo = re.sub(r'^C\s*', '', codigo, flags=re.IGNORECASE) # elimina 'C' + cualquier número de espacios al inicio
    codigo = str(codigo).strip()              # quita espacios al inicio y al final
    codigo = codigo.replace(' ', '')          # elimina cualquier espacio restante en el medio
    return codigo

#Conviertir a String y aplicar funcion para lipiar codigo
df_ventas['Cliente'] = df_ventas['Cliente'].apply(limpiar_codigo)
df_ventas

,Cliente,Fecha_Venta,Canal,Categoria,Venta
0,79520588,2025-06-01 00:00:00,Retail,ARTICULOS DE ASEO,75146.61
1,1094952285,2025-06-10 00:00:00,Retail,KIT MULTICATEGORÍA,54621.85
2,38553251,2025-07-18 00:00:00,Retail,CUIDADO CORPORAL,25887.37
3,72271652,2025-07-22 00:00:00,Retail,ARTICULOS DE ASEO,72352.94
4,41753037,2025-07-17 00:00:00,Retail,CUIDADO FACIAL,158193.28
...,...,...,...,...,...
40688,ElEditorSAS,2025-06-02,Shopify,CUIDADO CORPORAL,132767.40
40689,EmpresasPúblicasdeMedellín,2025-06-13,Shopify,CUIDADO MANOS,107768.48
40690,independiente,2025-06-14,Shopify,CUIDADO CORPORAL,106508.02
40691,NIT830004707,2025-07-10,Shopify,KIT MULTICATEGORIA,66383.70


In [7]:
# Cargar el archivo Excel
ruta_excel = 'PLAN 120.xlsx'
plan120_excel = pd.read_excel(ruta_excel,sheet_name=None)

In [13]:
# ▶️ Paso 4: Inicializar ExcelWriter y resumen
resultados = {}

for mes, df_nuevos in plan120_excel.items():
    # Normalizar columna cliente
    df_nuevos['Cliente'] = df_nuevos['Cliente'].astype(str).str.strip().str.upper()

    # Fecha mínima de recompra (día después del ingreso)
    fecha_inicio = pd.to_datetime(df_nuevos['Fecha_Ingreso'].min()) + pd.Timedelta(days=1)
    
    df_nuevos['Fecha_Ingreso'] = pd.to_datetime(df_nuevos['Fecha_Ingreso'])
    df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta'])

    # Ventas después del ingreso
    df_recompras = df_ventas[
        (df_ventas['Cliente'].isin(df_nuevos['Cliente'])) &
        (df_ventas['Fecha_Venta'] >= fecha_inicio)
    ]

    # Agregamos info útil
    resumen = df_recompras.groupby(['Cliente']).agg({
        'Fecha_Venta': 'min',  # primera recompra
        'Canal': pd.Series.nunique,  # número de canales usados
        'Categoria': lambda x: ', '.join(sorted(set(x.dropna()))),
        'Venta': 'sum'
    }).rename(columns={
        'Fecha_Venta': 'FechaPrimeraRecompra',
        'Canal': 'CanalesRecompra',
        'Categoria': 'CategoriasRecompra',
        'Venta': 'ValorTotalRecompra'
    }).reset_index()

    # Unir con base original de nuevos
    df_final = df_nuevos.merge(resumen, on='Cliente', how='left')

    # Guardar resultado para ese mes
    resultados[mes] = df_final


KeyError: 'Fecha_Ingreso'

In [ ]:
# # ▶️ Paso 5: Procesar cada hoja (mes)
# for mes, df_nuevos in hojas_clientes.items():
#     print(f'Procesando hoja: {mes}')
    
#     df_nuevos['Cliente'] = df_nuevos['Cliente'].astype(str).str.strip().str.replace('^C', '', regex=True)
#     df_nuevos['MesIngreso'] = mes
    
#     df_merged = df_ventas.merge(df_nuevos[['Cliente']], on='Cliente', how='inner')
#     df_merged['FechaPrimeraCompra'] = df_merged.groupby('Cliente')['Fecha'].transform('min')
    
#     # Recompras después de la primera
#     df_recompras = df_merged[df_merged['Fecha'] > df_merged['FechaPrimeraCompra']].copy()

#     df_metrics = df_recompras.groupby('Cliente').agg(
#         Fecha_Recompra=('Fecha', 'min'),
#         Num_Recompras=('Fecha', 'count'),
#         Total_Recomprado=('Venta', 'sum'),
#         Canales=('Canal', lambda x: ', '.join(x.dropna().unique())),
#         Categorias=('Categoria', lambda x: ', '.join(x.dropna().unique()))
#     ).reset_index()

#     df_metrics['MesIngreso'] = mes
#     df_metrics['FechaPrimeraCompra'] = df_metrics['Cliente'].map(
#         df_merged.set_index('Cliente')['FechaPrimeraCompra']
#     )
#     df_metrics['DiasHastaRecompra'] = (
#         df_metrics['Fecha_Recompra'] - df_metrics['FechaPrimeraCompra']
#     ).dt.days

#     # Clientes sin recompra
#     sin_recompra = df_nuevos[~df_nuevos['Cliente'].isin(df_metrics['Cliente'])].copy()
#     sin_recompra['FechaPrimeraCompra'] = df_merged.groupby('Cliente')['Fecha'].min()
#     sin_recompra['Fecha_Recompra'] = pd.NaT
#     sin_recompra['Num_Recompras'] = 0
#     sin_recompra['Total_Recomprado'] = 0
#     sin_recompra['Canales'] = None
#     sin_recompra['Categorias'] = None
#     sin_recompra['DiasHastaRecompra'] = None

#     df_final = pd.concat([
#         df_metrics[[
#             'Cliente', 'MesIngreso', 'FechaPrimeraCompra', 'Fecha_Recompra',
#             'Num_Recompras', 'Total_Recomprado', 'Canales', 'Categorias', 'DiasHastaRecompra'
#         ]],
#         sin_recompra[[
#             'Cliente', 'MesIngreso', 'FechaPrimeraCompra', 'Fecha_Recompra',
#             'Num_Recompras', 'Total_Recomprado', 'Canales', 'Categorias', 'DiasHastaRecompra'
#         ]]
#     ], ignore_index=True)

#     # Guardar hoja
#     df_final.to_excel(writer, sheet_name=mes[:31], index=False)

#     # KPI resumen
#     total = df_final.shape[0]
#     con_recompra = df_final[df_final['Num_Recompras'] > 0].shape[0]
#     total_venta = df_final['Total_Recomprado'].sum()
#     ticket_prom = df_final[df_final['Num_Recompras'] > 0]['Total_Recomprado'].mean()

#     resumen_kpi.append({
#         'MesIngreso': mes,
#         'TotalClientes': total,
#         'ConRecompra': con_recompra,
#         '%Recompra': round(con_recompra / total * 100, 2),
#         'TotalRecomprado': total_venta,
#         'TicketPromedio': round(ticket_prom, 2) if not pd.isna(ticket_prom) else 0
#     })